# STIC validation on Ameriflux sites

Validation of STIC using field data from the ameriflux database and landsat acquisitions

In [ ]:
import os

os.environ["NUMBA_DISABLE_JIT"] = "1"
# os.environ["NUMBA_DEVELOPER_MODE"] = "1"
# os.environ["NUMBA_DEBUG"] = "1"

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import pandas as pd

In [ ]:
import extra.plot as plt
from extra.metrics import generate_metrics_table
from extra.process import focus_one_point, init_stic, run_stic

## Input

In [ ]:
data_path = os.path.join(
    os.path.expanduser("~"), "data", "validation", "fl_data.csv"
)

## Read data

In [ ]:
data = pd.read_csv(data_path, parse_dates=["date", "date (utc)"])

In [ ]:
data

### Plot data

In [ ]:
plt.plot_data(data)

In [ ]:
plt.plot_scatter(data)

## Compute initialization of STIC model

In [ ]:
%%time
init_v1_3 = init_stic(data, version="1.3")
init_v1_3.attrs["label"] = "init STIC 1.3"

In [ ]:
init_v1_3.head()

In [ ]:
%%time
init_v1_4 = init_stic(data, version="1.4")
init_v1_4.attrs["label"] = "init STIC 1.4"

In [ ]:
init_v1_4.head()

In [ ]:
plt.plot_scatter([init_v1_3, init_v1_4], x="ec_le", y="le")

## Compute STIC model

In [ ]:
%%time
res_v1_3 = run_stic(data, version="1.3")

In [ ]:
res_v1_3

In [ ]:
%%time
res_v1_4 = run_stic(data, version="1.4")

In [ ]:
res_v1_4

In [ ]:
plt.plot_scatter([res_v1_3, res_v1_4], x="ec_le", y="le")

In [ ]:
plt.plot_scatter_with_colorbar([res_v1_3, res_v1_4], x="ec_le_closed", y="le")

In [ ]:
plt.plot_scatter_with_colorbar([init_v1_4, res_v1_4], x="ec_le_closed", y="le")

## Compute metrics

In [ ]:
metrics_v1_3 = generate_metrics_table(
    res_v1_3,
    variables=["le", "le_closed"],
    ranges={
        "le": [
            (float("-inf"), 200),
            (200, 400),
            (400, 600),
            (600, float("inf")),
        ],
        "le_closed": [
            (float("-inf"), 200),
            (200, 400),
            (400, 600),
            (600, float("inf")),
        ],
    },
)
metrics_v1_4 = generate_metrics_table(
    res_v1_4,
    variables=["le", "le_closed"],
    ranges={
        "le": [
            (float("-inf"), 200),
            (200, 400),
            (400, 600),
            (600, float("inf")),
        ],
        "le_closed": [
            (float("-inf"), 200),
            (200, 400),
            (400, 600),
            (600, float("inf")),
        ],
    },
)

In [ ]:
metrics_v1_3

In [ ]:
metrics_v1_4

In [ ]:
plt.plot_metrics(
    [metrics_v1_3, metrics_v1_4], "le_closed", "rmse", only_all=True
)

In [ ]:
plt.plot_metrics([metrics_v1_3, metrics_v1_4], "le", "rmse")

In [ ]:
plt.plot_metrics([metrics_v1_3, metrics_v1_4], "le_closed", "rmse")

In [ ]:
plt.plot_metrics([metrics_v1_3, metrics_v1_4], "le_closed", "nrmse_mean")

## Focus on a point

In [ ]:
focus_one_point(
    input_df=data,
    results_df=res_v1_4,
    name="US-Ton",
    date="2016-09-15",
    version="1.4",
)